# Module 10: Light Detective

## Mission Briefing

Your Pico is about to gain a superpower: it's going to **see light**. Well, not exactly "see" — but it will know whether a room is bright or dark, and react accordingly.

You'll wire up a **photoresistor** — a tiny component that changes how much electricity it lets through based on how much light hits it. Then you'll build an **automatic night light** that turns itself on when the room goes dark.

```
                    BRIGHT ROOM                          DARK ROOM
              ~~~~~~~~~~~~~~~~~~~~~~               .......................
              ~  +------------+    ~               .  +------------+    .
              ~  | Photoresist|    ~               .  | Photoresist|    .
    Light --> ~  |   (LDR)   |    ~     No light  .  |   (LDR)   |    .
              ~  +-----+------+   ~               .  +-----+------+    .
              ~        |          ~               .        |           .
              ~        v          ~               .        v           .
              ~   Low reading     ~               .   High reading     .
              ~   LED stays OFF   ~               .   LED turns ON!    .
              ~~~~~~~~~~~~~~~~~~~~~~               .......................
```

No switches. No buttons. The light itself is the trigger. You're building a circuit that **senses the world around it**.

---
## Science Spotlight: Light-Sensitive Materials

Some materials change how much electricity they let through based on how much light hits them. Your instructor will show you how a **voltage divider** — two resistors sharing voltage — turns this into a light sensor the Pico can read.

---
## Meet the Photoresistor (LDR)

A photoresistor — also called an **LDR** (Light Dependent Resistor) — looks like a small disc with a squiggly line on top. That squiggly line is the light-sensitive material.

```
        +-------------+
        |  /\/\/\/\   |    <-- squiggly pattern
        |  \/\/\/\/   |        (light-sensitive)
        +------+------+
               |   |
              Pin  Pin
           (no polarity — either direction works)
```

How it behaves:
- **Bright light** = LOW resistance (electricity flows easily)
- **Darkness** = HIGH resistance (electricity has a hard time flowing)

It has **no polarity** — you can plug it in either direction.

We'll also use a **10k ohm resistor** to build a **voltage divider**. This is how the Pico can read the light level as a number.

---
## Wiring: Voltage Divider + LED

This circuit has two parts: a **light sensor** (input) and an **LED** (output).

### Circuit Diagram

```
   Light Sensor (Voltage Divider):

   Pico 3.3V ──── wire ──── Photoresistor Pin 1
                             Photoresistor Pin 2 ──── junction ──── wire ──── Pico GP26 (ADC0)
                                                         |
                                                    [10k ohm Resistor]
                                                         |
                                                      Pico GND

   LED Circuit:
   Pico GP15 ──── [220 ohm Resistor] ──── LED (+) ──── LED (-) ──── GND
```

### Step-by-Step

1. **Place the photoresistor** on the breadboard — each of its 2 pins in a different row
2. **Connect a wire** from **3.3V** on the Pico to one pin of the photoresistor
3. **Connect a wire** from the **other pin** of the photoresistor to **GP26** on the Pico
4. **Place the 10k ohm resistor** with one leg in the **same row** as the photoresistor/GP26 junction, and the other leg in a different row
5. **Connect a wire** from the other leg of the 10k ohm resistor to **GND**
6. **Add an LED** to the breadboard (long leg = +, short leg = -)
7. **Add a 220 ohm resistor** connecting **GP15** to the LED's long leg row
8. **Connect** the LED's short leg row to **GND**

Plug in your USB cable.

**Important:** The junction point — where the photoresistor, 10k ohm resistor, and GP26 wire all meet — must be in the **same breadboard row**. This is the heart of the voltage divider!

---
## Code Along: Read the Light Level

Let's see what the photoresistor is telling the Pico. We'll use `ADC` to read the voltage at the junction point of our voltage divider.

Fill in the blank:

In [ ]:
from machine import ADC
import time

adc = ADC(_____)                # Which GPIO pin is our voltage divider junction on?

for i in range(20):
    light_value = adc.read_u16()  # Read a 16-bit value (0 to 65535)
    print("Light level:", light_value)
    time.sleep(0.5)

Run the code and try these while it prints values:

- What value do you see in **normal room light**? ____________
- **Cover the sensor with your hand** — what happens to the number? ____________
- **Shine your phone flashlight** on the sensor — what happens? ____________

The Pico is reading the light level! Higher number = more light. Lower number = less light.

---
## Code Along: Find Your Threshold

To build an automatic night light, we need to decide: **how dark is "dark enough" to turn on the LED?**

That's called a **threshold** — a dividing line between "bright" and "dark."

Run this code and experiment to find a good threshold for your room:

In [ ]:
from machine import ADC
import time

adc = ADC(26)

print("Cover the sensor slowly and watch the numbers drop...")
print("Find a number between 'bright' and 'dark' -- that's your threshold!")
print()

for i in range(30):
    light_value = adc.read_u16()
    bar = "#" * (light_value // 2000)    # Simple visual bar
    print("Light:", light_value, bar)
    time.sleep(0.5)

My threshold value: ____________

(A good starting point is usually around **20000-30000**, but every room and sensor is different!)

---
## Code Along: Automatic Night Light

Now the magic moment — let's make the LED turn on automatically when it gets dark, and turn off when it's bright.

Fill in the blanks:

In [ ]:
from machine import Pin, ADC
import time

adc = ADC(26)
led = Pin(_____, Pin.OUT)        # Which GPIO pin is our LED on?

THRESHOLD = _____                # Use the threshold you found!

while True:
    light_value = adc.read_u16()
    
    if light_value < THRESHOLD:  # Dark!
        led.value(_____)         # Turn LED ON (what value means ON?)
        print("Dark! LED ON  -- Light:", light_value)
    else:                        # Bright!
        led.value(_____)         # Turn LED OFF (what value means OFF?)
        print("Bright. LED off -- Light:", light_value)
    
    time.sleep(0.2)

Try it! Cover the sensor with your hand — the LED should light up. Remove your hand — the LED should turn off.

You just built an **automatic night light** — the same idea behind street lights that turn on at dusk!

---
## Code Along: Brightness Control with PWM

Instead of just ON or OFF, let's make the LED **brighter when it's darker** and **dimmer when it's lighter**. We'll use PWM for smooth brightness control.

The idea: the darker it is, the brighter the LED should glow — like a real adaptive night light.

Fill in the blanks:

In [ ]:
from machine import Pin, ADC, PWM
import time

adc = ADC(26)
led_pwm = PWM(Pin(_____))       # Which GPIO pin is our LED on?
led_pwm.freq(1000)

while True:
    light_value = adc.read_u16()
    
    # Invert: dark (low reading) should give HIGH brightness
    brightness = _____ - light_value   # What's the max ADC value?
    
    led_pwm.duty_u16(_____)      # What controls the LED brightness?
    print("Light:", light_value, "  Brightness:", brightness)
    time.sleep(0.1)

Cover the sensor slowly — the LED should gradually get brighter! Uncover it — the LED fades away.

The trick is **inverting** the value: `65535 - light_value`. When `light_value` is low (dark), the result is high (bright LED). When `light_value` is high (bright room), the result is low (dim LED).

---
## Experiments

Try these modifications and observations:

1. **Phone flashlight test:** Shine your phone flashlight directly on the sensor. What's the highest reading you can get? What about complete darkness (cover with both hands cupped tightly)?

2. **Distance test:** Hold your phone flashlight at different distances from the sensor. How does the reading change?

3. **Different thresholds:** In the night light code, try changing the threshold to a much higher or lower value. What happens? Can you make it so the LED is almost always on? Almost always off?

4. **Sensitivity adjustment:** Can you add a "dead zone" around the threshold so the LED doesn't flicker when the light is right at the boundary? (Hint: use two thresholds — one for turning on and one for turning off)

---
## Challenge: Dawn Simulator Alarm

Build a **dawn simulator** — when darkness is detected for more than **10 seconds**, the LED gradually brightens from off to full brightness, simulating a sunrise!

### How it works:
1. Continuously read the light level
2. If it's dark, start counting how long it's been dark
3. If it's been dark for 10+ seconds, **gradually increase LED brightness** from 0 to full over several seconds
4. If the room becomes bright again, reset everything

### Useful hints:

```python
# You can track time like this:
import time
dark_start = time.ticks_ms()        # Record when darkness began
elapsed = time.ticks_diff(time.ticks_ms(), dark_start)  # How long in ms

# To gradually increase brightness:
for brightness in range(0, 65536, 655):  # 100 steps from 0 to ~65535
    led_pwm.duty_u16(brightness)
    time.sleep(0.05)                      # Total fade: ~5 seconds
```

Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Photoresistor (LDR)** | A Light Dependent Resistor — its resistance changes based on how much light hits it |
| **Voltage Divider** | Two resistors sharing voltage — the voltage at the junction changes when one resistor changes |
| **Threshold** | A cutoff value used to make a decision (e.g., below this number = "dark") |
| **ADC (Analog-to-Digital Converter)** | Hardware that turns a smooth voltage into a digital number (0-65535) |
| **Resistance** | How much a material resists the flow of electricity — measured in ohms |
| **Inverting** | Flipping a value — turning a low number into a high number and vice versa (e.g., `65535 - value`) |
| **Sensor** | A component that detects something in the physical world (light, temperature, motion, etc.) |

---
*Circuit Explorers — Module 10: Light Detective*